##### Copyright 2019 The TensorFlow Authors.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

This notebook was developed starting from one of google's Tensorflow tutorials.

We will compare feed forward neural network with [Convolutional Neural Network](https://developers.google.com/machine-learning/glossary/#convolutional_neural_network) (CNN) to classify MNIST digits.

### Import TensorFlow

In [ ]:
import tensorflow as tf

from tensorflow.keras import datasets, layers, models

### Download and prepare the MNIST dataset

In [ ]:
(train_images, train_labels), (test_images, test_labels) = datasets.mnist.load_data()

train_images = train_images.reshape((60000, 28, 28, 1))
test_images = test_images.reshape((10000, 28, 28, 1))

# Normalize pixel values to be between 0 and 1
train_images, test_images = train_images / 255.0, test_images / 255.0

In [ ]:
# Visualize data
import matplotlib.pyplot as plt

plotData = test_images[66]
plotData = plotData.reshape(28, 28)
plt.gray() # use this line if you don't want to see it in color
plt.imshow(plotData)
plt.show()

# DNN approach first

In [ ]:
train_images_DNN = train_images.reshape((60000, 784))
test_images_DNN = test_images.reshape((10000, 784))

# We are going to build the model in a slightly different way
model_DNN = models.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(10, activation='softmax'),
])

model_DNN.summary()

model_DNN.compile(optimizer=tf.keras.optimizers.Adam(
                                               learning_rate=0.005),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history_DNN = model_DNN.fit(train_images_DNN, train_labels, epochs=5,
                             validation_split=0.1)

In [ ]:
test_loss, test_acc = model_DNN.evaluate(test_images_DNN, test_labels)

pred_DNN = list(model_DNN.predict(test_images_DNN))

import pandas as pd
probs_DNN = pd.DataFrame(pred_DNN)

#Let's look at the outputs of the linear and NN discriminants
fig, ax = plt.subplots(figsize=(5, 5))

probs_DNN.plot(ax=ax, kind='hist', bins=100, title='predicted probabilities', alpha=0.7, label='DNN')
ax.legend(frameon=False, prop={'size': 16})

In [ ]:
from sklearn.metrics import roc_curve

# convert test labels to binary response
my_resp = []
for el in test_labels:
  if el == 0:
    my_resp.append(1)
  else:
    my_resp.append(0)

# plot ROC for classifying 0s
fig, ax = plt.subplots()
fpr2, tpr2, _ = roc_curve(my_resp, probs_DNN[0].values)
ax.plot(fpr2, tpr2, label='DNN')

ax.set_title('ROC curve', size='20')
ax.set_xlabel('False positive rate', size='20')
ax.set_ylabel('True positive rate', size='20')
ax.set_xlim(0,)
ax.set_ylim(0,)
ax.set_aspect('equal')

ax.legend(frameon=False, prop={'size': 16})


## DNN training history

In [ ]:
def plot_history(history, title=''):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(title, size=14)

    ax1.plot(history.history['loss'], label='Train')
    ax1.plot(history.history['val_loss'], label='Validation')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Loss vs Epoch')
    ax1.legend(frameon=False)
    ax1.grid(True)

    ax2.plot(history.history['accuracy'], label='Train')
    ax2.plot(history.history['val_accuracy'], label='Validation')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Accuracy vs Epoch')
    ax2.legend(frameon=False)
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

plot_history(history_DNN, title='DNN training history')

# Convolutional Neural Networks

### Create the convolutional base

The 6 lines of code below define the convolutional base using a common pattern: a stack of [Conv2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D) and [MaxPooling2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPool2D) layers.

As input, a CNN takes tensors of shape (image_height, image_width, color_channels), ignoring the batch size. If you are new to color channels, MNIST has one (because the images are grayscale), whereas a color image has three (R,G,B). In this example, we will configure our CNN to process inputs of shape (28, 28, 1), which is the format of MNIST images. We do this by passing the argument `input_shape` to our first layer.



In [ ]:
model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
])

Let display the architecture of our model so far.

In [ ]:
model.summary()

Above, you can see that the output of every Conv2D and MaxPooling2D layer is a 3D tensor of shape (height, width, channels). The width and height dimensions tend to shrink as we go deeper in the network. The number of output channels for each Conv2D layer is controlled by the first argument (e.g., 32 or 64). Typically,  as the width and height shrink, we can afford (computationally) to add more output channels in each Conv2D layer.

### Add Dense layers on top
To complete our model, we will feed the last output tensor from the convolutional base (of shape (3, 3, 64)) into one or more Dense layers to perform classification. Dense layers take vectors as input (which are 1D), while the current output is a 3D tensor. First, we will flatten (or unroll) the 3D output to 1D,  then add one or more Dense layers on top. MNIST has 10 output classes, so we use a final Dense layer with 10 outputs and a softmax activation.

In [ ]:
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(10, activation='softmax'))

 Here's the complete architecture of our model.

In [ ]:
model.summary()

As you can see, our (3, 3, 64) outputs were flattened into vectors of shape (576) before going through two Dense layers.

### Compile and train the model

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(
                                               learning_rate=0.005),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history_CNN = model.fit(train_images, train_labels, epochs=5,
                        validation_split=0.1)

### Evaluate the model

In [ ]:
test_loss, test_acc = model.evaluate(test_images, test_labels)

In [ ]:
print(test_acc)

As you can see, our simple CNN has achieved a test accuracy of over 99%. Not bad for a few lines of code!

## CNN training history

In [ ]:
plot_history(history_CNN, title='CNN training history')

## Confusion matrices

A confusion matrix shows how often each true class is predicted as each other class. Bright cells on the diagonal mean good performance; off-diagonal cells reveal systematic mis-classifications.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def plot_confusion(model, images, labels, title=''):
    preds = np.argmax(model.predict(images, verbose=0), axis=1)
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(7, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=list(range(10)))
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, size=13)
    plt.tight_layout()
    plt.show()

plot_confusion(model_DNN, test_images_DNN, test_labels, title='DNN – confusion matrix')
plot_confusion(model,     test_images,     test_labels, title='CNN – confusion matrix')

## Misclassified examples

Let's look at some digits the CNN got wrong to get an intuition for the failure modes.

In [ ]:
cnn_preds = np.argmax(model.predict(test_images, verbose=0), axis=1)
wrong_idx = np.where(cnn_preds != test_labels)[0]

n_show = 16
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
fig.suptitle('CNN – misclassified examples', size=13)
for ax, idx in zip(axes.flat, wrong_idx[:n_show]):
    ax.imshow(test_images[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f'true={test_labels[idx]}, pred={cnn_preds[idx]}', size=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f'Total misclassified: {len(wrong_idx)} / {len(test_labels)} '
      f'({100*len(wrong_idx)/len(test_labels):.2f}%)')